# Pocket Generation

Providing accurate pocket information to the pharmacophore modeler is crucial;
otherwise it would generate features that are outside of the binding pocket.
A pocket is defined as a boolean array representing the voxelized volume of the pocket,
and is used by the modeler to only select field points that lie within the pocket.

In [1]:
import numpy as np     # To generate mock data
import t2fpharm_study  # To obtain a pre-generated chemical system
import t2fpharm

## From Pre-Computed Data

A pocket can be generated from pre-computed data.
This requires providing spatial information
defining the grid on which the pocket voxels are sampled,
as well as the corresponding boolean array defining the pocket volume.
For example:

In [2]:
# A 20x30x40 grid starting at coordinates (0, 0, 0) with 0.3 Å spacing
GRID_ORIGIN = (0, 0, 0)
GRID_SHAPE = (20, 30, 40)
GRID_SPACING = 0.3

# Pocket voxels as a binary array with the same shape as the grid
POCKET_VOXELS = np.ones(GRID_SHAPE)

In [3]:
# Create the grid
pocket_grid: t2fpharm.grid.Grid = t2fpharm.grid.from_anchor_shape_spacing(
    anchor=GRID_ORIGIN,
    shape=GRID_SHAPE,
    spacing=GRID_SPACING,
    anchor_type="lower",
)
# Create the field
pocket_from_data: t2fpharm.pocket.Pocket = t2fpharm.pocket.from_tensor(
    tensor=POCKET_VOXELS,
    grid=pocket_grid,
)

A `Grid` object can also be generated from various other input data.
For more information, see the notebook [`4_grid.ipynb`](./4_grid.ipynb).

## From Ligand

A pocket can be generated from a receptor–ligand complex
by expanding the ligand volume and voxelization.
For this, a receptor–ligand complex is required as a `System` object.
Here, we use a pre-generated object for brevity;
for more information on creating a `System`, see the notebook [`2_system.ipynb`](./2_system.ipynb).

In [4]:
rcomplex: t2fpharm.system.System = t2fpharm_study.manager().complex("1aq1")

A pocket can be generated by specifying the ligand and pocket generation parameters:

In [5]:
# Specify residue name, chain ID, and residue sequence number of the ligand of interest
LIGAND_RES_NAME: str = "STU"
LIGAND_CHAIN_ID: str = "A"
LIGAND_RES_SEQ: int = 299

# Select ligand atoms
atoms = rcomplex.composition.atoms
ligand_mask = (
    (atoms["res_name"] == LIGAND_RES_NAME) &
    (atoms["chain_id"] == LIGAND_CHAIN_ID) &
    (atoms["res_seq"] == LIGAND_RES_SEQ)
)

# Define pocket specs
RADII_OFFSET: float = 2.7  # Expansion radius (in Å) for each atom
OPENING_RADIUS: float = 1  # Radius (in Å) of the morphological opening operation to remove small artifacts
GRID_SPACING: float = 0.3  # Grid spacing (in Å)

# Create pocket
pocket_from_ligand: t2fpharm.pocket.Pocket = t2fpharm.pocket.from_ligand(
    system=rcomplex,
    ligand_mask=ligand_mask,
    ligand_radii_offset=RADII_OFFSET,  
    opening_radius=OPENING_RADIUS,  
    grid=GRID_SPACING,  
)

More information about the pocket generation algorithm can be found in the docstring:

In [6]:
help(t2fpharm.pocket.from_ligand)

Help on function from_ligand in module caddpy.pocket:

from_ligand(system: 'ChemicalSystem', ligand_mask: 'ArrayLike', ligand_radii: 'ArrayLike | None' = None, ligand_radii_offset: 'float | Sequence[float]' = 2.5, erosion_radius: 'float' = 0, opening_radius: 'float' = 0, morphology_order: "tuple[Literal['opening', 'erosion'], Literal['opening', 'erosion']]" = ('opening', 'erosion'), grid: 'float | Sequence[float] | Grid' = 0.3, trim: 'bool' = True) -> 'Pocket'
    Create a pocket from a ligand.

    This function works as follows:
    1. Ligand atoms are selected from the `system` using `ligand_mask`.
    2. The ligand volume is converted to a voxel grid using the provided `grid`,
       where each ligand atom is represented by a sphere of radius `ligand_radii + ligand_radii_offset`.
    3. The receptor volume is converted to a voxel grid using the same `grid`.
    4. The pocket voxels are determined as the voxels that are occupied by the ligand
       but not occupied by the receptor.

## From DoGSite

Pockets can also be generated using the DoGSite algorithms from the ProteinsPlus webserver:

In [7]:
pockets_from_dogsite = t2fpharm.pocket.from_dogsite(
    system=rcomplex, 
    ligand_id=(LIGAND_RES_NAME, LIGAND_CHAIN_ID, LIGAND_RES_SEQ), 
    include_subpockets=True, 
    calculate_druggability=True, 
    algorithm="scorer"  # or "3" to use DoGSite3 instead of DoGSiteScorer
)

This returns a `Pockets` object, holding information
about all of the detected pockets:

In [8]:
pockets_from_dogsite.pockets[:5]

,label,volume,point_count,is_subpocket,parent_label,pocket
label,,,,,,
1,1,1817.664324,28401,False,1,"Field(\n grid=Grid(\n shape=[69 76 52],\n ..."
2,2,364.800065,5700,False,2,"Field(\n grid=Grid(\n shape=[37 36 32],\n ..."
3,3,273.920049,4280,False,3,"Field(\n grid=Grid(\n shape=[25 35 19],\n ..."
4,4,218.112039,3408,False,4,"Field(\n grid=Grid(\n shape=[29 22 25],\n ..."
5,5,201.856036,3154,False,5,"Field(\n grid=Grid(\n shape=[15 25 28],\n ..."


In [9]:
pockets_from_dogsite.external_data[:5]

,name,lig_cov,poc_cov,lig_name,volume,enclosure,surface,depth,surf/vol,lid/hull,...,simpleScore,drugScore,center_x,center_y,center_z,max_radius,atom_serials,mrc,label,parent_label
label,,,,,,,,,,,,,,,,,,,,,
1,P_0,90.48,20.45,STU_A_299,1817.66,0.14,2520.56,23.25,1.386706,-,...,0.68,0.814071,-7.95,27.4,12.98,18.61,"(163, 165, 166, 167, 169, 172, 176, 182, 185, ...",MrcFile(\n n_xyz = [73 80 56]\n mode ...,1,1
2,P_1,0.0,0.0,STU_A_299,364.8,0.14,643.89,12.94,1.765049,-,...,0.18,0.621417,17.42,24.08,21.55,12.95,"(1444, 1451, 1454, 1457, 1488, 1505, 1507, 150...",MrcFile(\n n_xyz = [41 40 36]\n mode ...,2,2
3,P_2,0.0,0.0,STU_A_299,273.92,0.2,523.72,14.06,1.911945,-,...,0.09,0.566932,-8.7,26.06,19.86,10.46,"(214, 215, 217, 219, 235, 239, 241, 242, 2047,...",MrcFile(\n n_xyz = [29 39 23]\n mode ...,3,3
4,P_3,0.0,0.0,STU_A_299,218.11,0.22,529.43,12.68,2.427353,-,...,0.13,0.5,17.87,25.53,10.5,8.92,"(1386, 1390, 1403, 1404, 1405, 1463, 1465, 146...",MrcFile(\n n_xyz = [33 26 29]\n mode ...,4,4
5,P_4,0.0,0.0,STU_A_299,201.86,0.25,374.86,9.13,1.85703,-,...,0.06,0.348162,-0.5,40.31,31.25,8.28,"(1883, 1884, 1898, 1900, 1902, 1903, 1904, 194...",MrcFile(\n n_xyz = [19 29 32]\n mode ...,5,5


In [10]:
pockets_from_dogsite.display()

ThemeManager()

NGLWidget(gui_style='ngl')

We can then select a pocket; for example based on ligand and pocket coverage:

In [11]:
selected_label = pockets_from_dogsite.external_data.sort_values(
    ["lig_cov", "poc_cov"],
    ascending=False
).iloc[0]["label"]
pocket_from_dogsite = pockets_from_dogsite.pockets.loc[selected_label, "pocket"]

## Methods and Attributes

`Pocket` is a subclass of `Field`, and thus provides all methods and attributes of `Field` objects (For more information, see the notebook [`5_field.ipynb`](./5_field.ipynb)). Additional methods and attributes specific to `Pocket` objects are discussed below.

Visualize the pocket:

In [12]:
pocket_from_ligand.display()

NGLWidget(gui_style='ngl')

In [13]:
pocket_from_dogsite.display()

NGLWidget(gui_style='ngl')

Receptor atoms that make up the pocket:

In [14]:
pocket_from_ligand.atoms

,chain_id,res_name,res_seq,i_code,res_poly,res_std,serial,name,alt_loc,occupancy,temp_factor,element,charge,element_index
serial,,,,,,,,,,,,,,
161,A,ILE,10,,True,True,161,N,,1,0,N,<NA>,6
163,A,ILE,10,,True,True,163,CA,,1,0,C,<NA>,5
164,A,ILE,10,,True,True,164,HA,,1,0,H,<NA>,0
165,A,ILE,10,,True,True,165,C,,1,0,C,<NA>,5
166,A,ILE,10,,True,True,166,O,,1,0,O,<NA>,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2417,A,LEU,148,,True,True,2417,CG,,1,0,C,<NA>,5
2419,A,LEU,148,,True,True,2419,CD1,,1,0,C,<NA>,5
2420,A,LEU,148,,True,True,2420,HD11,,1,0,H,<NA>,0


Check whether the pocket volume has holes (i.e., `False` values completely surrounded by `True` values):

In [15]:
pocket_from_dogsite.holes().any()

Array(True, dtype=bool)

In [16]:
pocket_from_ligand.holes().any()

Array(False, dtype=bool)

Calculate the coverage of ligand by the pocket:

In [17]:
ligand_atom_coords = rcomplex.trajectory.points[ligand_mask.to_numpy()]

In [18]:
pocket_from_ligand.point_coverage(ligand_atom_coords)

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True], dtype=bool)

In [19]:
pocket_from_dogsite.point_coverage(ligand_atom_coords)

Array([ True,  True, False, False, False, False, False, False,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False, False,  True, False,
        True, False, False, False, False, False, False], dtype=bool)

As can be seen, the DoGSite pocket has holes and also does not cover all ligand atoms,
which is problematic since we can't use it to generate an accurate ligand pharmacophore from the complex.